# San Diego 311 Analysis - Updated Findings

This notebook consolidates current results from:
- `01_citywide_data_preprocessing_eda.ipynb`
- `02_council_district_eda.ipynb`
- `03_catboost_modeling.ipynb`


## Executive Summary

- District-level disparity is still visible in aggregate: **Districts 5 and 6 show ~1.60x longer waits** in the district comparison output.
- Composition explains a large share of that gap: Districts 5 and 6 show **2.45x higher infrastructure-request concentration** in the infrastructure split output.
- Pavement maintenance remains a major service-level gap: **243 days vs 146 days** median (Districts 5 and 6 vs other districts).
- Mixed-effects modeling supports the structural hypothesis that lower-share neighborhoods tend to be slower.
- CatBoost performance improved materially after stronger feature engineering and tuning, with optimized test MAE near **8.00 days** in the latest saved run.


## Citywide Findings (From 01 Notebook)

### Data and Trend Context
- Multi-year closed-request data was cleaned and standardized into a unified citywide analysis table.
- Feature engineering added temporal and request-context predictors used downstream.

### Association Patterns
- Infrastructure-related request categories show stronger association with longer response time.
- More recent years generally align with faster response time relative to earlier periods.
- Calendar-only effects are weaker than service-mix effects in explaining wait-time variation.


## District and Equity Findings (From 02 Notebook)

### Aggregate Gap
- Districts 5 and 6: approximately **1.60x longer** waits in the group comparison output.

### Composition Effect
- Districts 5 and 6: approximately **2.45x more infrastructure requests** in the infrastructure composition output.

### Service-Specific Gap
- Pavement maintenance median response time:
  - Districts 5 and 6: **243 days**
  - Other districts: **146 days**

Interpretation: a meaningful portion of overall disparity is composition-driven, with a strong service-line exception in pavement maintenance.


## Mixed-Effects Model (From 02 Notebook)

### Specification
- Outcome: `log_wait = log1p(case_age_days)`
- Cross-classified random effects: neighborhood (`comm_plan_name`) and request type (`service_name`)
- Fixed controls include month effects and neighborhood request-share predictor

### Main Result
- `neigh_share` coefficient: **-1.032** (SE 0.190, z = -5.436, p < 0.001, 95% CI [-1.404, -0.660])

Interpretation: higher neighborhood request share is associated with shorter expected response time after controls.


## Predictive Modeling Results (From 03 Notebook)

### Baselines and Model Progression
- Median baseline (saved output):
  - R² (days): **-0.091**
  - RMSE (days): **20.77**
  - MAE (days): **9.71**

- Baseline CatBoost (saved output):
  - R² (days): **0.208**
  - RMSE (days): **17.69**
  - MAE (days): **8.371**

- Optimized CatBoost (saved output):
  - R² (days): **0.258**
  - RMSE (days): **17.127**
  - MAE (days): **8.001**
  - Log metrics: R² **0.485**, RMSE **0.944**, MAE **0.698**

### Modeling Notes
- Time-based train/validation/test split was used throughout.
- Largest gains came from feature engineering (cleaned categorical/context features) plus tuning.


## Challenges Encountered and Resolutions

### 1) Mixed-effects convergence instability
- Issue: non-positive-definite Hessian under over-parameterized specs.
- Resolution: removed duplicated fixed/random structure for the same grouping variable and kept cross-classified random effects clean.

### 2) Categorical type errors in CatBoost
- Issue: float/NaN values in `cat_features` columns.
- Resolution: consistent missing-value handling and string casting for categorical predictors.

### 3) Rolling-feature alignment and reproducibility
- Issue: index alignment errors and run-to-run variance from ordering.
- Resolution: deterministic sorting, index-safe rolling feature construction, and controlled training settings.


## Current Conclusion

The combined evidence supports a composition-driven explanation for much of the district-level gap, while also identifying service-specific inequity in pavement maintenance. Mixed-effects estimates show a robust negative neighborhood-share relationship with wait time, and CatBoost modeling now provides materially better predictive accuracy than naive baseline prediction.
